# Breast Cancer Prediction: Machine Learning + Streamlit (Google Colab)

Run the cells from top to bottom (**Runtime > Run all**). The last two cells start the Streamlit app and give you a public link.

> Educational project only. Not a medical device.

## 1. Install

In [ ]:
!pip -q install streamlit

## 2. Explore the data

In [ ]:
import matplotlib.pyplot as plt, pandas as pd, seaborn as sns
from sklearn.datasets import load_breast_cancer

df = load_breast_cancer(as_frame=True).frame
mean_cols = [c for c in df.columns if c.startswith("mean ")]
df["diagnosis"] = df["target"].map({0: "Malignant", 1: "Benign"})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["diagnosis"].value_counts().plot.bar(ax=axes[0], color=["#5BA96B", "#E4572E"], rot=0)
axes[0].set_title("Class balance")
sns.heatmap(df[mean_cols].corr(), cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlation of the 10 mean features")
plt.tight_layout(); plt.show()

df.groupby("diagnosis")[mean_cols].mean().T.round(3)

## 3. Training script
Writes `train_model.py` (identical to the VS Code version), then runs it. It compares Logistic Regression, Random Forest and Gradient Boosting and saves `model_artifacts.joblib`.

In [ ]:
%%writefile train_model.py
"""
Breast Cancer Prediction - Model Training
=========================================
Dataset : Breast Cancer Wisconsin (Diagnostic) - bundled with scikit-learn,
          so no download or CSV file is needed.
Target  : 0 = Malignant, 1 = Benign
Inputs  : the 10 "mean" measurements of the cell nuclei (radius, texture, ...)

Trains 3 models, compares them with 5-fold cross-validation, evaluates on a
held-out test set and saves everything the Streamlit app needs.

Run:  python train_model.py
Output: model_artifacts.joblib
"""

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# 1. Load data -----------------------------------------------------------
raw = load_breast_cancer(as_frame=True)
df = raw.frame
mean_cols = [c for c in df.columns if c.startswith("mean ")]
X = df[mean_cols]
y = df["target"]                       # 0 = malignant, 1 = benign
print(f"Dataset: {X.shape[0]} patients, {X.shape[1]} features")
print("Class counts:", y.value_counts().rename({0: 'malignant', 1: 'benign'}).to_dict())

# 2. Train / test split --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# 3. Models (scaling lives inside the pipeline, so no data leakage) -------
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
metrics = {}
for name, model in models.items():
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    metrics[name] = {
        "cv_accuracy": float(cv_acc.mean()),
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred)),
        "recall": float(recall_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    print(f"{name:<20} CV acc {cv_acc.mean():.3f} | test acc {metrics[name]['accuracy']:.3f}"
          f" | ROC-AUC {metrics[name]['roc_auc']:.3f}")

best_name = max(metrics, key=lambda n: metrics[n]["cv_accuracy"])
print(f"\nBest model (by cross-validation): {best_name}")

# 4. Feature importance of the tree model ---------------------------------
importance = pd.Series(models["Random Forest"].feature_importances_,
                       index=mean_cols).sort_values(ascending=False)

# 5. Save everything the app needs ----------------------------------------
artifacts = {
    "models": models,
    "best_model_name": best_name,
    "metrics": metrics,
    "feature_names": mean_cols,
    "feature_ranges": {c: (float(X[c].min()), float(X[c].max()), float(X[c].mean()))
                       for c in mean_cols},
    "feature_importance": importance.to_dict(),
    "class_names": {0: "Malignant", 1: "Benign"},
    "sample_data": df[mean_cols + ["target"]].sample(150, random_state=RANDOM_STATE),
}
joblib.dump(artifacts, "model_artifacts.joblib")
print("Saved model_artifacts.joblib")


In [ ]:
!python train_model.py

## 4. Streamlit app
Writes `app.py` (identical to the VS Code version).

In [ ]:
%%writefile app.py
"""
Breast Cancer Prediction - Streamlit App
========================================
Run:  streamlit run app.py
Needs model_artifacts.joblib in the same folder (create it with: python train_model.py)

Educational project only - NOT a medical device.
"""

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Breast Cancer Predictor", page_icon="🩺", layout="wide")


@st.cache_resource
def load_artifacts():
    return joblib.load("model_artifacts.joblib")


art = load_artifacts()
models = art["models"]
metrics = art["metrics"]
features = art["feature_names"]
ranges = art["feature_ranges"]
class_names = art["class_names"]

# ---------------- Sidebar ----------------
st.sidebar.header("Settings")
model_name = st.sidebar.selectbox(
    "Choose a model", list(models.keys()),
    index=list(models.keys()).index(art["best_model_name"]))
model = models[model_name]
st.sidebar.markdown("**Test accuracy**")
for n, m in metrics.items():
    star = " ⭐" if n == art["best_model_name"] else ""
    st.sidebar.write(f"- {n}: {m['accuracy']:.1%}{star}")
st.sidebar.info("Educational project. Not for real medical decisions.")

st.title("🩺 Breast Cancer Prediction")
st.write("Predict whether a tumour is **benign** or **malignant** from cell-nucleus "
         "measurements (Breast Cancer Wisconsin dataset, 569 patients).")

tab_predict, tab_perf, tab_data = st.tabs(["🔮 Predict", "📊 Model performance", "🗂 Data"])

# ---------------- Tab 1: Predict ----------------
with tab_predict:
    st.subheader("Enter the measurements")
    use_example = st.radio("Start from", ["Average patient", "Typical malignant", "Typical benign"],
                           horizontal=True)
    sample = art["sample_data"]
    if use_example == "Typical malignant":
        defaults = sample[sample["target"] == 0][features].median()
    elif use_example == "Typical benign":
        defaults = sample[sample["target"] == 1][features].median()
    else:
        defaults = pd.Series({f: ranges[f][2] for f in features})

    values = {}
    cols = st.columns(3)
    for i, f in enumerate(features):
        lo, hi, _ = ranges[f]
        step = float((hi - lo) / 200)
        with cols[i % 3]:
            values[f] = st.slider(f.replace("mean ", "Mean ").title(), float(lo), float(hi),
                                  float(np.clip(defaults[f], lo, hi)), step=step,
                                  key=f"{use_example}-{f}")

    if st.button("Predict", type="primary"):
        X_new = pd.DataFrame([values])[features]
        pred = int(model.predict(X_new)[0])
        proba = model.predict_proba(X_new)[0]
        label = class_names[pred]
        (st.success if pred == 1 else st.error)(
            f"Prediction: **{label}**  (confidence {proba[pred]:.1%})")
        c1, c2 = st.columns(2)
        c1.metric("Malignant probability", f"{proba[0]:.1%}")
        c2.metric("Benign probability", f"{proba[1]:.1%}")
        st.progress(float(proba[1]), text="Benign probability")

# ---------------- Tab 2: Performance ----------------
with tab_perf:
    st.subheader("Model comparison (held-out test set)")
    table = pd.DataFrame(metrics).T[["cv_accuracy", "accuracy", "precision", "recall", "f1", "roc_auc"]]
    table.columns = ["CV accuracy", "Test accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    st.dataframe(table.style.format("{:.3f}").highlight_max(axis=0, color="#cfe8d5"),
                 use_container_width=True)

    left, right = st.columns(2)
    with left:
        st.markdown(f"**Confusion matrix: {model_name}**")
        cm = np.array(metrics[model_name]["confusion_matrix"])
        fig, ax = plt.subplots(figsize=(4, 3.5))
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1], ["Malignant", "Benign"])
        ax.set_yticks([0, 1], ["Malignant", "Benign"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center",
                    color="white" if v > cm.max() / 2 else "black", fontsize=14)
        st.pyplot(fig)
    with right:
        st.markdown("**Feature importance (Random Forest)**")
        imp = pd.Series(art["feature_importance"]).sort_values()
        fig, ax = plt.subplots(figsize=(4.5, 3.5))
        ax.barh([i.replace("mean ", "") for i in imp.index], imp.values, color="#0e7c86")
        ax.set_xlabel("Importance")
        st.pyplot(fig)

# ---------------- Tab 3: Data ----------------
with tab_data:
    st.subheader("Sample of the dataset")
    show = art["sample_data"].copy()
    show["diagnosis"] = show["target"].map(class_names)
    st.dataframe(show.drop(columns="target").head(50), use_container_width=True)
    st.markdown("**Class balance in the sample**")
    st.bar_chart(show["diagnosis"].value_counts())


## 5. Launch the app
Run the next cell, then the one after it. Open the `loca.lt` link and, if asked for a password, paste the IP printed by the first cell.

In [ ]:
import subprocess, time, urllib.request

# start Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501",
                  "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

# localtunnel asks for this IP address as the "password" on first visit
print("Tunnel password (public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
print("Open the https://....loca.lt link printed below.\nThis cell keeps running while the app is in use; stop it to shut the app down.")

In [ ]:
!npx -y localtunnel --port 8501